In [1]:
box::use(
  dada = dada2,
  bios = Biostrings,
  shor = ShortRead,
  stri = stringr,
  msa,
  ape,
  tidyverse[...], ggplot2[...], ggtree[...]
)

In [2]:
wdpath <- getwd()
demult.path <- paste0(wdpath, "/zuzana/demultiplexed/")
fns <- sort(list.files(demult.path, pattern = ".fastq.gz", full.names = TRUE))

In [3]:

filts <- file.path(wdpath, "zuzana/filtered", basename(fns))

track <- dada$filterAndTrim(
  fns, filts,
  minLen = 2000, # Min length for sequence
  maxLen = 3000, # Max length for sequence
  rm.phix = FALSE, # No phix added (Illumina specific)
  qualityType = "FastqQuality", # Suggested for PacBio
  multithread = TRUE, # Allow multithread
  verbose = FALSE, # Print progress
  maxEE = 2, # Suggested default
  minQ = 20
)
exists <- file.exists(filts)
paste("Sample", basename(filts[!exists]), "did not pass filtering")
write.csv(track, "zuzana/zuzana_2000_filtering_tracking.csv")

Creating output directory: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered



[1] "Sample  did not pass filtering"

In [4]:
filts <- file.path(wdpath, "zuzana/filtered", basename(fns))
data.frame(
  "barcode" = sapply(
    filts[exists],
    function(x) {stri$str_split_i(basename(x), "[.]", 3)}
  ),
  "median" = sapply(
    filts[exists],
    function(x) {median(bios$width(bios$readDNAStringSet(x ,format='FASTQ')))}
  )
) |> write.csv("zuzana/zuzana_filtered_medians.csv")

In [5]:
filts <- file.path(wdpath, "zuzana/filtered", basename(fns))
exists <- file.exists(filts)
filts <- filts[exists]
drp <- dada$derepFastq(filts, verbose = TRUE)

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered/A_fennica_Af2_1A_cut.fastq.gz

Encountered 87 unique sequences from 558 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered/A_koskei_Sc705_5P_cut.fastq.gz

Encountered 59 unique sequences from 567 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered/A_koskei_WV658_1H_cut.fastq.gz

Encountered 56 unique sequences from 467 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered/A_koskei_WV685_1F_cut.fastq.gz

Encountered 100 unique sequences from 1101 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered/A_trappei_CU132_5P_cut.fastq.gz

Encountered 69 unique sequences from 350 total sequences 

In [ ]:
err <- dada$learnErrors(
  drp,
  errorEstimationFunction = dada$PacBioErrfun, # Set error estimation function for PacBio 
  multithread = TRUE, # Allow multithread
  BAND_SIZE = 32 # Suggested for Pacbio
)

101712897 total bases in 43135 reads from 59 samples will be used for learning the error rates.


In [7]:
dd <- dada$dada(
  drp,
  err = err,
  BAND_SIZE = 32,
  multithread = TRUE
)

Sample 1 - 558 reads in 87 unique sequences.
Sample 2 - 567 reads in 59 unique sequences.
Sample 3 - 467 reads in 56 unique sequences.
Sample 4 - 1101 reads in 100 unique sequences.
Sample 5 - 350 reads in 69 unique sequences.
Sample 6 - 1515 reads in 303 unique sequences.
Sample 7 - 446 reads in 213 unique sequences.
Sample 8 - 603 reads in 309 unique sequences.
Sample 9 - 3356 reads in 745 unique sequences.
Sample 10 - 457 reads in 189 unique sequences.
Sample 11 - 916 reads in 115 unique sequences.
Sample 12 - 507 reads in 79 unique sequences.
Sample 13 - 180 reads in 44 unique sequences.
Sample 14 - 438 reads in 102 unique sequences.
Sample 15 - 875 reads in 80 unique sequences.
Sample 16 - 144 reads in 99 unique sequences.
Sample 17 - 357 reads in 63 unique sequences.
Sample 18 - 101 reads in 35 unique sequences.
Sample 19 - 502 reads in 148 unique sequences.
Sample 20 - 246 reads in 125 unique sequences.
Sample 21 - 135 reads in 78 unique sequences.
Sample 22 - 898 reads in 191 u

In [8]:
data.frame(
  "denoised" = sapply(
    dd,
    function(x) {sum(dada$getUniques(x))}
  )
) |> write.csv("zuzana/zuzana_denoised.csv")

In [9]:
st <- dada$makeSequenceTable(dd)
dim(st)

[1]  61 261

In [10]:
st.nobim <- dada$removeBimeraDenovo(
  st, method = "consensus",
  multithread = TRUE,
  verbose = TRUE
)

Identified 72 bimeras out of 261 input sequences.



In [11]:
data.frame(
  "nobimeras" = rowSums(st.nobim)
) |> write.csv("zuzana/zuzana_nobim.csv")

In [12]:
asv_table <- t(st.nobim)
colnames(asv_table) <- sapply(strsplit(rownames(st.nobim), "[.]"), `[`, 1)
rep_seqs <- bios$DNAStringSet(colnames(st.nobim))
rownames(asv_table) <- paste0("zuzana_ASV_", seq(colnames(st.nobim)))
names(rep_seqs) <- rownames(asv_table)

In [13]:
row.names(asv_table)
colnames(asv_table)

[1] "zuzana_ASV_1"   "zuzana_ASV_2"   "zuzana_ASV_3"   "zuzana_ASV_4"  
  [5] "zuzana_ASV_5"   "zuzana_ASV_6"   "zuzana_ASV_7"   "zuzana_ASV_8"  
  [9] "zuzana_ASV_9"   "zuzana_ASV_10"  "zuzana_ASV_11"  "zuzana_ASV_12" 
 [13] "zuzana_ASV_13"  "zuzana_ASV_14"  "zuzana_ASV_15"  "zuzana_ASV_16" 
 [17] "zuzana_ASV_17"  "zuzana_ASV_18"  "zuzana_ASV_19"  "zuzana_ASV_20" 
 [21] "zuzana_ASV_21"  "zuzana_ASV_22"  "zuzana_ASV_23"  "zuzana_ASV_24" 
 [25] "zuzana_ASV_25"  "zuzana_ASV_26"  "zuzana_ASV_27"  "zuzana_ASV_28" 
 [29] "zuzana_ASV_29"  "zuzana_ASV_30"  "zuzana_ASV_31"  "zuzana_ASV_32" 
 [33] "zuzana_ASV_33"  "zuzana_ASV_34"  "zuzana_ASV_35"  "zuzana_ASV_36" 
 [37] "zuzana_ASV_37"  "zuzana_ASV_38"  "zuzana_ASV_39"  "zuzana_ASV_40" 
 [41] "zuzana_ASV_41"  "zuzana_ASV_42"  "zuzana_ASV_43"  "zuzana_ASV_44" 
 [45] "zuzana_ASV_45"  "zuzana_ASV_46"  "zuzana_ASV_47"  "zuzana_ASV_48" 
 [49] "zuzana_ASV_49"  "zuzana_ASV_50"  "zuzana_ASV_51"  "zuzana_ASV_52" 
 [53] "zuzana_ASV_53"  "zuzana_ASV_54"  "zuzana_ASV_55"  "zuzana_ASV_56" 
 [57] "zuzana_ASV_57"  "zuzana_ASV_58"  "zuzana_ASV_59"  "zuzana_ASV_60" 
 [61] "zuzana_ASV_61"  "zuzana_ASV_62"  "zuzana_ASV_63"  "zuzana_ASV_64" 
 [65] "zuzana_ASV_65"  "zuzana_ASV_66"  "zuzana_ASV_67"  "zuzana_ASV_68" 
 [69] "zuzana_ASV_69"  "zuzana_ASV_70"  "zuzana_ASV_71"  "zuzana_ASV_72" 
 [73] "zuzana_ASV_73"  "zuzana_ASV_74"  "zuzana_ASV_75"  "zuzana_ASV_76" 
 [77] "zuzana_ASV_77"  "zuzana_ASV_78"  "zuzana_ASV_79"  "zuzana_ASV_80" 
 [81] "zuzana_ASV_81"  "zuzana_ASV_82"  "zuzana_ASV_83"  "zuzana_ASV_84" 
 [85] "zuzana_ASV_85"  "zuzana_ASV_86"  "zuzana_ASV_87"  "zuzana_ASV_88" 
 [89] "zuzana_ASV_89"  "zuzana_ASV_90"  "zuzana_ASV_91"  "zuzana_ASV_92" 
 [93] "zuzana_ASV_93"  "zuzana_ASV_94"  "zuzana_ASV_95"  "zuzana_ASV_96" 
 [97] "zuzana_ASV_97"  "zuzana_ASV_98"  "zuzana_ASV_99"  "zuzana_ASV_100"
[101] "zuzana_ASV_101" "zuzana_ASV_102" "zuzana_ASV_103" "zuzana_ASV_104"
[105] "zuzana_ASV_105" "zuzana_ASV_106" "zuzana_ASV_107" "zuzana_ASV_108"
[109] "zuzana_ASV_109" "zuzana_ASV_110" "zuzana_ASV_111" "zuzana_ASV_112"
[113] "zuzana_ASV_113" "zuzana_ASV_114" "zuzana_ASV_115" "zuzana_ASV_116"
[117] "zuzana_ASV_117" "zuzana_ASV_118" "zuzana_ASV_119" "zuzana_ASV_120"
[121] "zuzana_ASV_121" "zuzana_ASV_122" "zuzana_ASV_123" "zuzana_ASV_124"
[125] "zuzana_ASV_125" "zuzana_ASV_126" "zuzana_ASV_127" "zuzana_ASV_128"
[129] "zuzana_ASV_129" "zuzana_ASV_130" "zuzana_ASV_131" "zuzana_ASV_132"
[133] "zuzana_ASV_133" "zuzana_ASV_134" "zuzana_ASV_135" "zuzana_ASV_136"
[137] "zuzana_ASV_137" "zuzana_ASV_138" "zuzana_ASV_139" "zuzana_ASV_140"
[141] "zuzana_ASV_141" "zuzana_ASV_142" "zuzana_ASV_143" "zuzana_ASV_144"
[145] "zuzana_ASV_145" "zuzana_ASV_146" "zuzana_ASV_147" "zuzana_ASV_148"
[149] "zuzana_ASV_149" "zuzana_ASV_150" "zuzana_ASV_151" "zuzana_ASV_152"
[153] "zuzana_ASV_153" "zuzana_ASV_154" "zuzana_ASV_155" "zuzana_ASV_156"
[157] "zuzana_ASV_157" "zuzana_ASV_158" "zuzana_ASV_159" "zuzana_ASV_160"
[161] "zuzana_ASV_161" "zuzana_ASV_162" "zuzana_ASV_163" "zuzana_ASV_164"
[165] "zuzana_ASV_165" "zuzana_ASV_166" "zuzana_ASV_167" "zuzana_ASV_168"
[169] "zuzana_ASV_169" "zuzana_ASV_170" "zuzana_ASV_171" "zuzana_ASV_172"
[173] "zuzana_ASV_173" "zuzana_ASV_174" "zuzana_ASV_175" "zuzana_ASV_176"
[177] "zuzana_ASV_177" "zuzana_ASV_178" "zuzana_ASV_179" "zuzana_ASV_180"
[181] "zuzana_ASV_181" "zuzana_ASV_182" "zuzana_ASV_183" "zuzana_ASV_184"
[185] "zuzana_ASV_185" "zuzana_ASV_186" "zuzana_ASV_187" "zuzana_ASV_188"
[189] "zuzana_ASV_189"

[1] "A_fennica_Af2_1A_cut"            "A_koskei_Sc705_5P_cut"          
 [3] "A_koskei_WV658_1H_cut"           "A_koskei_WV685_1F_cut"          
 [5] "A_trappei_CU132_5P_cut"          "Ac_koskei_WV685_1A_cut"         
 [7] "Ac_koskei_WV685_1B_cut"          "Ac_koskei_WV685_1C_cut"         
 [9] "Ac_morrowiae_BR983A_1A_cut"      "Ac_morrowiae_BR983A_1B_cut"     
[11] "Ac_morrowiae_BR983A_5A_cut"      "Am_gerdemannii_ON205A_1B_cut"   
[13] "Am_gerdemannii_ON205A_1B_SV_cut" "Am_gerdemannii_ON205A_1C_SV_cut"
[15] "C_corymbiforme_Blask5_1A_cut"    "C_corymbiforme_Blask5_1B_cut"   
[17] "C_corymbiforme_Blask5_1C_cut"    "C_luteum_SA101_5S_cut"          
[19] "D_heterogama_BEG35_1M_cut"       "D_heterogama_BEG35_1N_cut"      
[21] "D_heterogama_BEG35_1O_cut"       "De_heterogama_BR154_1A_cut"     
[23] "De_heterogama_BR154_1B_cut"      "De_heterogama_BR154_1C_cut"     
[25] "De_heterogama_IL203A_1A_cut"     "De_heterogama_IL203A_1B_cut"    
[27] "De_heterogama_IL203A_1C_cut"     "Di_epigaea_KS210_1A_cut"        
[29] "Di_epigaea_KS210_1C_cut"         "Di_epigaea_KS210_5A_cut"        
[31] "E_infrequens_Blask3_1F_cut"      "F_mosseae_BEG161_10A_cut"       
[33] "F_mosseae_BEG161_1A_cut"         "F_mosseae_BEG161_5B_cut"        
[35] "F_mosseae_BEG95_5A_cut"          "Fu_mosseae_BEG12_1A_cut"        
[37] "Fu_mosseae_BEG12_1B_cut"         "Fu_mosseae_BEG12_1C_cut"        
[39] "G_rosea_BEG9_1Na_ cut"           "G_rosea_BEG9_1Nb_cut"           
[41] "Gi_rosea_BR155B_1A_cut"          "Gi_rosea_BR155B_1B_cut"         
[43] "Gi_rosea_BR155B_1C_cut"          "Gi_rosea_KS885_1A_cut"          
[45] "Gi_rosea_KS885_1B_cut"           "Gi_rosea_KS885_1C_cut"          
[47] "P_laccatum_AH960_55_cut"         "P_occultum_CU126_10A_cut"       
[49] "P_occultum_CU126_10B_cut"        "P_occultum_CU126_5B_cut"        
[51] "R_irregularis_PH5_1A_cut"        "Ra_gregaria_BEG243_1A_cut"      
[53] "Ra_gregaria_BEG243_1B_cut"       "Ra_gregaria_BEG243_1C_cut"      
[55] "Racocetra_sp_179_Blask4_10A_cut" "S_calospora_IL209_1A_cut"       
[57] "S_calospora_IL209_1C_cut"        "S_cerrevisiae_Yeast_cut"        
[59] "S_viscosum_AH179_15_1A_cut"      "S_viscosum_AH179_15_1B_cut"     
[61] "S_viscosum_AH179_15_1C_cut"

In [14]:
bios$writeXStringSet(rep_seqs, "zuzana/zuzana_rep_seqs.fasta")
write.table(
  asv_table,
  "zuzana/zuzana_asv_table.tsv",
  sep = "\t",
  row.names = TRUE,
  col.names = NA,
  quote = FALSE
)